<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/Grouped_Query_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
class GroupedQueryAttention(nn.Module):
  def __init__(self,d_in,d_out,dropout,num_heads,num_kv_groups,dtype=None,qkv_bias=False):
    super().__init__()
    assert d_out%num_heads ==0,"d_out must be divisble by num_heads"
    assert num_heads % num_kv_groups == 0 ,"num_heads must be divisble by num_kv_groups"
    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads

    self.W_key = nn.Linear(d_in,num_kv_groups*self.head_dim,bias=qkv_bias,dtype=dtype)
    self.W_value = nn.Linear(d_in,num_kv_groups*self.head_dim,bias=qkv_bias,dtype=dtype)
    self.num_kv_groups = num_kv_groups
    self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias,dtype=dtype)
    self.dropout = nn.Dropout(dropout)
    self.group_size = num_heads//num_kv_groups
    self.out_proj = nn.Linear(d_out,d_out,dtype=dtype)

    self.register_buffer("cache_k",None,persistent=False)
    self.register_buffer("cache_v",None,persistent=False)
    self.ptr_current_pos = 0
  def forward(self,x,use_cache=False):
    b,num_tokens,_ = x.shape

    queries = self.W_query(x) #(b,num_tokens,num_head*head_dim)
    values = self.W_value(x) #(b,num_tokens,num_kv_groups*head_dim)
    keys = self.W_key(x) #(b,num_tokens,num_kv_groups*head_dim)

    queries = queries.view(b,num_tokens,self.num_heads,self.head_dim).transpose(1,2)
    keys_new = keys.view(b,num_tokens,self.num_kv_groups,self.head_dim).transpose(1,2)
    values_new = values.view(b,num_tokens,self.num_kv_groups,self.head_dim).transpose(1,2)

    if use_cache:
      if self.cache_k is None:
        self.cache_k,self.cache_v = keys_new,values_new
      else:
        self.cache_k = torch.cat((self.cache_k,keys_new),dim=2)
        self.cache_v = torch.cat((self.cache_v,values_new),dim=2)
      keys_base,value_base = self.cache_k,self.cache_v
    else:
      keys_base,values_base = keys_new,values_new
      if self.cache_k is not None or self.cache_v is not None:
        self.cache_k = None
        self.cache_v = None
        self.ptr_current_pos = 0
    keys = keys_base.repeat_interleave(self.group_size,dim=2)
    values = values_base.repeat_interleave(self.group_size,dim=2)

    attn_scores = queries @ keys.transpose(2,3)
    #Causal mask
    num_tokens_q = queries.shape[-2]
    num_tokens_k  = keys.shape[-2]
    device = queries.device
    if use_cache:
      q_positions = torch.arange(
          self.ptr_current_pos,self.ptr_current_pos+num_tokens_q,device=device,dtype=torch.long
      )
    else:
      q_positions = torch.arange(
          num_tokens_q,device=device,dtype=torch.long
      )
    k_positions = torch.arange(
        num_tokens_k,device=device,dtype=torch.long
    )
    mask = q_positions.unsqueeze(-1) < k_positions.unsqueeze(0)
    attn_scores = attn_scores.masked_fill(mask,-torch.inf)
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim=-1)
    assert keys.shape[-1] == self.head_dim
    attn_weights = self.dropout(attn_weights)
    attn_output = attn_weights @ values
    attn_output = attn_output.transpose(1,2)
    context_vec = attn_output.contiguous().view(b,num_tokens,self.d_out)
    context_vec = self.out_proj(context_vec)
    return context_vec
  def reset_cache(self):
    self.cache_k,self.cache_v = None,None
    self.ptr_current_pos = 0